# Otimização de Infraestrutura na Fábrica de Paçoca: Árvores Geradoras Mínimas (MST)
## Otimização de Leitos de Tubulação de Utilidades e Redes de Automação SCADA via Algoritmos de Kruskal e Prim

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Linha de Produção de Paçoca  
**Equipe:** Grupo 7  
**Perfil:** Engenharia de Controle e Automação (Matemática Discreta & Árvores Geradoras Mínimas)

---

### 1. Contexto Industrial e Engenharia de Automação
No projeto eletromecânico e na infraestrutura de automação da **Linha de Produção de Paçoca** (beneficiamento de amendoim, torra, dosagem, moagem, prensagem e embalagem), os custos com **infraestrutura de eletrocalhas industriais** (cabeamento de rede SCADA PROFINET RT / Modbus TCP) e **leitos de tubulação de utilidades (*pipe racks*)** representam uma parcela expressiva do custo de capital (*CAPEX*) e de manutenção (*OPEX*).

As utilidades industriais compartilhadas e as redes de comunicação precisam interligar com máxima confiabilidade e mínima extensão física os principais ativos dos quatro setores da planta:
- **Linha de Ar Comprimido (6 a 8 bar):** acionamento pneumático dos ejetores ultrarrápidos da selecionadora óptica, cilindros de prensagem (`PRN-401`), válvulas de desvio (`XV-301..304`) e cabeçotes de selagem da embaladora *Flow-Pack* (`M-401`).
- **Linha de Vapor Industrial / Gás Combustível (GN/GLP):** alimentação dos queimadores do forno contínuo de torra (`FOR-201`) e trocadores do secador de leito fluidizado (`SEC-101`).
- **Barramento SCADA e Segurança Funcional:** anel de controle e telemetria ligando os CLPs setoriais (CLP 01 ao CLP 04) e os instrumentos ISA-5.1 à sala de controle central.

O objetivo formal da modelagem via **Teoria dos Grafos** é determinar a **Árvore Geradora Mínima (Minimum Spanning Tree - MST)**. A MST assegura que todos os equipamentos da fábrica estejam plenamente integrados à malha de utilidades e ao barramento de controle com o menor comprimento linear possível de canaletas/cabos, eliminando laços redundantes desnecessários.

#### Modelagem Matemática Formal
Considera-se o grafo ponderado e não direcionado $G = (V, E, w)$:
- $V$: Conjunto de vértices representando os equipamentos operacionais e painéis da planta ($|V| = n$).
- $E$: Conjunto de arestas representando os corredores físicos, eletrocalhas e canaletas viáveis para instalação ($|E| = m$).
- $w: E \to \mathbb{R}^+$: Função de ponderação que associa a cada canaleta $e \in E$ a sua distância física euclidiana/construtiva em metros $w(e)$.

Uma Árvore Geradora $T = (V, E_T)$ é um subgrafo conexo e acíclico que abrange todos os vértices de $V$, com $|E_T| = |V| - 1$. A MST busca minimizar a função de custo linear total:

$$W(T) = \sum_{e \in E_T} w(e)$$

---
### 2. Fundamentação dos Algoritmos de Otimização Topológica

#### A. Algoritmo de Kruskal (Abordagem Gulosa Centrada em Arestas)
- **Princípio Operacional:** Ordena todas as arestas viáveis em ordem crescente de peso. A cada iteração, adiciona a aresta de menor comprimento à solução, desde que suas extremidades pertençam a componentes conexas distintas (não formando ciclo).
- **Estrutura de Dados:** Utiliza **Conjuntos Disjuntos (*Disjoint Set Union - DSU / Union-Find*)** com as otimizações de *Compressão de Caminho (Path Compression)* e *União por Rank (Union by Rank)*.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |E|)$ devido à etapa de ordenação das arestas.

#### B. Algoritmo de Prim (Abordagem Gulosa Centrada em Vértices / Fronteira de Corte)
- **Princípio Operacional:** Faz crescer uma única árvore a partir de um vértice raiz inicial $r \in V$ (por exemplo, a recepção de matéria-prima ou o painel central). Em cada passo, examina a fronteira de corte $(S, V \setminus S)$ e anexa a aresta de menor peso incidente a um vértice ainda não visitado.
- **Estrutura de Dados:** Utiliza uma **Fila de Prioridades (*Min-Heap*)** para extrair dinamicamente a aresta ótima de corte.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |V|)$ com *Binary Heap*.

In [1]:
# Importação de módulos essenciais de estruturas de dados e visualização
from typing import List, Tuple, Dict, Set
import heapq
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Configuração de estilo visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas científicas e de engenharia carregadas com sucesso.")

Bibliotecas científicas e de engenharia carregadas com sucesso.


---
### 3. Implementação Orientada a Objetos: Estruturas de Dados e Algoritmos Puros

Implementamos a classe `DisjointSet` (Union-Find) com otimizações assintóticas completas e as funções determinísticas dos algoritmos de **Kruskal** e **Prim** com tipagem estática (*type hints*) e registros de auditoria operacional.

In [2]:
class DisjointSet:
    """
    Estrutura de Dados Disjoint Set Union (Union-Find) para detecção de ciclos:
    - Compressão de Caminho (Path Compression): achata a árvore durante consultas.
    - União por Rank (Union by Rank): acopla a árvore menor à raiz da árvore maior.
    """
    def __init__(self, vertices: List[str]):
        self.parent: Dict[str, str] = {v: v for v in vertices}
        self.rank: Dict[str, int] = {v: 0 for v in vertices}

    def find(self, item: str) -> str:
        """Encontra o representante da componente com compressão de caminho."""
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, root1: str, root2: str) -> bool:
        """Une dois conjuntos disjuntos. Retorna True se uniu, False se já no mesmo conjunto."""
        r1 = self.find(root1)
        r2 = self.find(root2)

        if r1 == r2:
            return False  # Ciclo detectado: mesmo representante

        if self.rank[r1] < self.rank[r2]:
            self.parent[r1] = r2
        elif self.rank[r1] > self.rank[r2]:
            self.parent[r2] = r1
        else:
            self.parent[r2] = r1
            self.rank[r1] += 1
        return True


def kruskal_mst(vertices: List[str], edges: List[Tuple[str, str, float]]) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """
    Executa o Algoritmo de Kruskal sobre o grafo da planta de processo.
    Retorna as arestas que compõem a MST, o custo total linear e o log detalhado de execução.
    """
    # 1. Ordenação gulosa baseada em peso
    sorted_edges = sorted(edges, key=lambda x: x[2])
    dsu = DisjointSet(vertices)

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []

    for u, v, w in sorted_edges:
        root_u = dsu.find(u)
        root_v = dsu.find(v)

        if root_u != root_v:
            dsu.union(u, v)
            mst_edges.append((u, v, w))
            total_weight += w
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "INCLUÍDA",
                "Motivo": "Vértices em componentes separadas"
            })
        else:
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "REJEITADA",
                "Motivo": "Formaria ciclo fechado redundante"
            })

        if len(mst_edges) == len(vertices) - 1:
            # Condição de parada ótima: MST com |V| - 1 arestas atingida
            break

    return mst_edges, total_weight, execution_log


def prim_mst(vertices: List[str], edges: List[Tuple[str, str, float]], start_node: str) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """
    Executa o Algoritmo de Prim utilizando Min-Heap (fila de prioridade).
    Retorna as arestas da MST, o custo linear total e o histórico de expansão da árvore.
    """
    adj: Dict[str, List[Tuple[float, str]]] = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))

    visited: Set[str] = {start_node}
    min_heap: List[Tuple[float, str, str]] = []  # Tupla: (peso, u, v)

    # Enfileira arestas incidentes ao nó inicial
    for w, neighbor in adj[start_node]:
        heapq.heappush(min_heap, (w, start_node, neighbor))

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    step = 1

    while min_heap and len(visited) < len(vertices):
        w, u, v = heapq.heappop(min_heap)

        if v in visited:
            execution_log.append({
                "Iteração": step,
                "Aresta Analisada": f"{u} - {v}",
                "Peso (m)": w,
                "Decisão": "IGNORADA",
                "Vértice Destino": v,
                "Motivo": f"{v} já integrado à arborescência"
            })
            step += 1
            continue

        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w

        execution_log.append({
            "Iteração": step,
            "Aresta Analisada": f"{u} - {v}",
            "Peso (m)": w,
            "Decisão": "ADICIONADA",
            "Vértice Destino": v,
            "Motivo": f"Expande malha para {v}"
        })
        step += 1

        for next_w, neighbor in adj[v]:
            if neighbor not in visited:
                heapq.heappush(min_heap, (next_w, v, neighbor))

    return mst_edges, total_weight, execution_log

---
### 4. Definição do Grafo da Fábrica de Paçoca (Grupo 7)

O layout industrial da fábrica de paçoca do Grupo 7 é estruturado em 4 setores de processamento, integrando 8 ativos mecânicos principais conectados por 14 corredores viáveis de canaletas e leitos:

| Vértice | Tag de Ativo | Setor Operacional | Descrição do Equipamento Industrial |
| :---: | :---: | :--- | :--- |
| $v_1$ | `SILO-101` | Setor 100 — Recepção | Silo de Espera de Amendoim Cru (`LT-101`) |
| $v_2$ | `SEC-101` | Setor 100 — Pré-Tratamento | Secador Rotativo de Leito Fluidizado (`TT-101`, `FS-101`) |
| $v_3$ | `FOR-201` | Setor 200 — Tratamento Térmico | Forno Contínuo de Torra a Gás (`TT-201`, `TS-201`, `XV-201`) |
| $v_4$ | `DES-202` | Setor 200 — Limpeza Fina | Rolos Despeliculadores e Ciclone de Extração (`M-202`) |
| $v_5$ | `MAN-301` | Setor 300 — Dosagem | Manifold de Dosagem Gravimétrica (`WT-301`, `WT-302`, `WT-303`) |
| $v_6$ | `MOI-301` | Setor 300 — Moagem | Moinho de Martelos Industrial (`M-301`) |
| $v_7$ | `HOM-301` | Setor 300 — Homogeneização | Misturador Homogeneizador Helicoidal de Massa (`HOM-301`) |
| $v_8$ | `PRN-401` | Setor 400 — Prensagem & Envase | Prensa Rotativa de Compactação e Embalagem (`PS-401`, `M-401`) |

In [3]:
# Vértices da Planta (Equipamentos Principais dos Setores 100 a 400)
nodes = [
    "SILO-101", "SEC-101", "FOR-201", "DES-202",
    "MAN-301", "MOI-301", "HOM-301", "PRN-401"
]

# Rotas Viáveis de Eletrocalhas e Pipe Racks com Distâncias Físicas em Metros
edges = [
    ("DES-202", "MAN-301", 10.0),  # Duto de grãos torrados e linha de utilidades S200-S300
    ("FOR-201", "DES-202", 12.0),  # Interligação térmica de processo forno-despeliculador
    ("MAN-301", "MOI-301", 12.0),  # Alimentação direta da dosagem para os moinhos de martelo
    ("HOM-301", "PRN-401", 14.0),  # Transporte da massa homogênea para a prensa compactadora
    ("SILO-101", "SEC-101", 15.0), # Alimentação de grãos cru do silo para o secador de leito
    ("DES-202", "MOI-301", 15.0),  # Canaleta de utilidades transversal entre S200 e S300
    ("MOI-301", "HOM-301", 16.0),  # Duto de pasta de amendoim moída para misturador
    ("FOR-201", "MOI-301", 18.0),  # Leito de ar seco e cabos de segurança de acionamento
    ("SEC-101", "FOR-201", 20.0),  # Rota de grãos secos pré-aquecidos para a torra contínua
    ("MAN-301", "HOM-301", 22.0),  # Linha auxiliar de dosagem direta de aditivos/glucose
    ("SEC-101", "DES-202", 25.0),  # Canaleta lateral de retorno de ar e utilidades gerais
    ("MOI-301", "PRN-401", 28.0),  # Leito de intertravamento de segurança entre moagem e prensa
    ("FOR-201", "MAN-301", 30.0),  # Rota periférica de cabeamento e instrumentação
    ("SILO-101", "DES-202", 35.0)  # Tronco principal de ar comprimido do galpão de insumos
]

total_initial_length = sum(w for _, _, w in edges)
print(f"Grafo da Fábrica de Paçoca carregado: {len(nodes)} Equipamentos e {len(edges)} Canaletas de Infraestrutura Viáveis.")
print(f"Metragem total pré-otimização (somatório de todas as rotas): {total_initial_length:.2f} metros lineares.")

Grafo da Fábrica de Paçoca carregado: 8 Equipamentos e 14 Canaletas de Infraestrutura Viáveis.
Metragem total pré-otimização (somatório de todas as rotas): 247.00 metros lineares.


---
### 5. Execução, Validação e Equivalência Estrita dos Algoritmos

In [4]:
# Execução do Algoritmo de Kruskal
kruskal_edges, kruskal_weight, kruskal_log = kruskal_mst(nodes, edges)

# Execução do Algoritmo de Prim (Início na Recepção: SILO-101)
prim_edges, prim_weight, prim_log = prim_mst(nodes, edges, start_node="SILO-101")

economy = (1.0 - (kruskal_weight / total_initial_length)) * 100.0

print("=" * 70)
print(" RESULTADOS DA OTIMIZAÇÃO TOPOLÓGICA — SCADA-CORE FÁBRICA DE PAÇOCA")
print("=" * 70)
print(f"Metragem Total de Infraestrutura (Kruskal): {kruskal_weight:.2f} metros")
print(f"Metragem Total de Infraestrutura (Prim):    {prim_weight:.2f} metros")
print(f"Equivalência Estrita de Custo (MST):       {kruskal_weight == prim_weight}")
print(f"Economia Linear Obtida em Relação à Malha: {economy:.2f}%")
print(f"Número de Arestas Selecionadas (|V| - 1):   {len(kruskal_edges)} canaletas")
print("=" * 70)

 RESULTADOS DA OTIMIZAÇÃO TOPOLÓGICA — SCADA-CORE FÁBRICA DE PAÇOCA
Metragem Total de Infraestrutura (Kruskal): 99.00 metros
Metragem Total de Infraestrutura (Prim):    99.00 metros
Equivalência Estrita de Custo (MST):       True
Economia Linear Obtida em Relação à Malha: 59.92%
Número de Arestas Selecionadas (|V| - 1):   7 canaletas


#### Tabela 1: Histórico Passo a Passo de Execução do Algoritmo de Kruskal

In [5]:
df_kruskal = pd.DataFrame(kruskal_log)
df_kruskal

,Passo,Aresta,Peso (m),Status,Motivo
0,1,DES-202 - MAN-301,10.0,INCLUÍDA,Vértices em componentes separadas
1,2,FOR-201 - DES-202,12.0,INCLUÍDA,Vértices em componentes separadas
2,3,MAN-301 - MOI-301,12.0,INCLUÍDA,Vértices em componentes separadas
3,4,HOM-301 - PRN-401,14.0,INCLUÍDA,Vértices em componentes separadas
4,5,SILO-101 - SEC-101,15.0,INCLUÍDA,Vértices em componentes separadas
5,6,DES-202 - MOI-301,15.0,REJEITADA,Formaria ciclo fechado redundante
6,7,MOI-301 - HOM-301,16.0,INCLUÍDA,Vértices em componentes separadas
7,8,FOR-201 - MOI-301,18.0,REJEITADA,Formaria ciclo fechado redundante
8,9,SEC-101 - FOR-201,20.0,INCLUÍDA,Vértices em componentes separadas


#### Tabela 2: Histórico Passo a Passo de Expansão do Algoritmo de Prim

In [6]:
df_prim = pd.DataFrame(prim_log)
df_prim

,Iteração,Aresta Analisada,Peso (m),Decisão,Vértice Destino,Motivo
0,1,SILO-101 - SEC-101,15.0,ADICIONADA,SEC-101,Expande malha para SEC-101
1,2,SEC-101 - FOR-201,20.0,ADICIONADA,FOR-201,Expande malha para FOR-201
2,3,FOR-201 - DES-202,12.0,ADICIONADA,DES-202,Expande malha para DES-202
3,4,DES-202 - MAN-301,10.0,ADICIONADA,MAN-301,Expande malha para MAN-301
4,5,MAN-301 - MOI-301,12.0,ADICIONADA,MOI-301,Expande malha para MOI-301
5,6,DES-202 - MOI-301,15.0,IGNORADA,MOI-301,MOI-301 já integrado à arborescência
6,7,MOI-301 - HOM-301,16.0,ADICIONADA,HOM-301,Expande malha para HOM-301
7,8,HOM-301 - PRN-401,14.0,ADICIONADA,PRN-401,Expande malha para PRN-401


---
### 6. Visualização Gráfica da Topologia Industrial e Malha Otimizada MST

Geramos a representação gráfica do chão de fábrica da fábrica de paçoca, posicionando os ativos conforme as coordenadas reais dos setores fabris (Setor 100 ao Setor 400). As rotas redundantes descartadas são exibidas em linha tracejada vermelha, enquanto os trechos selecionados para a Árvore Geradora Mínima (MST) são destacados em linhas sólidas com caixas de medição linear.

In [7]:
# Construção do Grafo no NetworkX
G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Coordenadas Físicas 2D dos Ativos no Chão de Fábrica (Metros no Layout Fabril)
pos = {
    "SILO-101": (10.0, 32.0), # Setor 100 - Recepção
    "SEC-101":  (24.0, 32.0), # Setor 100 - Secador
    "FOR-201":  (40.0, 32.0), # Setor 200 - Forno de Torra
    "DES-202":  (40.0, 16.0), # Setor 200 - Despeliculador
    "MAN-301":  (56.0, 32.0), # Setor 300 - Manifold de Dosagem
    "MOI-301":  (56.0, 16.0), # Setor 300 - Moinho de Martelos
    "HOM-301":  (72.0, 24.0), # Setor 300 - Homogeneizador
    "PRN-401":  (88.0, 24.0), # Setor 400 - Prensa e Embalagem
}

# Conjuntos de Arestas da MST e Arestas Descartadas
mst_set = set((u, v) for u, v, _ in kruskal_edges) | set((v, u) for u, v, _ in kruskal_edges)
mst_edges_plot = [(u, v) for u, v in G.edges() if (u, v) in mst_set]
discarded_edges_plot = [(u, v) for u, v in G.edges() if (u, v) not in mst_set]

# Paleta de cores por setor industrial
node_colors = [
    "#3b82f6", "#3b82f6",  # S100: Azul (SILO-101, SEC-101)
    "#f59e0b", "#f59e0b",  # S200: Âmbar/Laranja (FOR-201, DES-202)
    "#10b981", "#10b981", "#10b981",  # S300: Verde (MAN-301, MOI-301, HOM-301)
    "#8b5cf6"              # S400: Roxo (PRN-401)
]

fig, ax = plt.subplots(figsize=(15, 8.5), dpi=120)

# 1. Desenha arestas redundantes descartadas (vermelho tracejado)
nx.draw_networkx_edges(
    G, pos, edgelist=discarded_edges_plot,
    edge_color="#ef4444", style="dashed", width=1.8, alpha=0.65,
    label="Rotas Descartadas (Evitam Ciclos)", ax=ax
)

# 2. Desenha arestas da MST selecionadas (verde esmeralda sólido)
nx.draw_networkx_edges(
    G, pos, edgelist=mst_edges_plot,
    edge_color="#059669", style="solid", width=3.8, alpha=0.95,
    label="Malha Mínima Otimizada (MST — 99.00 m)", ax=ax
)

# 3. Desenha nós dos equipamentos industriais
nx.draw_networkx_nodes(
    G, pos, nodelist=nodes, node_color=node_colors,
    node_size=2800, edgecolors="#0f172a", linewidths=2.2, ax=ax
)

# 4. Rótulos dos nós
nx.draw_networkx_labels(
    G, pos, font_size=9.5, font_weight="bold",
    font_color="#ffffff", font_family="sans-serif", ax=ax
)

# 5. Rótulos de peso nas arestas selecionadas
edge_labels = { (u, v): f"{d['weight']:.0f}m" for u, v, d in G.edges(data=True) if (u, v) in mst_set }
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=edge_labels, font_size=9, font_weight="bold",
    bbox=dict(boxstyle="round,pad=0.25", facecolor="#ecfdf5", edgecolor="#059669", alpha=0.95), ax=ax
)

ax.set_title("Otimização Topológica de Leitos de Utilidades e Redes SCADA — Fábrica de Paçoca (Grupo 7)", fontsize=14, fontweight="bold", pad=18)
ax.text(0.5, 1.01, f"Metragem Total: {kruskal_weight:.2f} m | Economia de Infraestrutura: {economy:.1f}% em relação ao grafo completo ({total_initial_length:.0f} m)",
        transform=ax.transAxes, ha="center", fontsize=11, color="#334155", fontweight="semibold")

# Legenda explicativa dos setores industriais
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="#059669", lw=3.5, label="Canaleta/Leito MST (Selecionada)"),
    Line2D([0], [0], color="#ef4444", lw=2, linestyle="--", label="Rota Descartada (Ciclo)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#3b82f6", markersize=11, label="Setor 100: Recepção e Secagem"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#f59e0b", markersize=11, label="Setor 200: Torra e Despeliculagem"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#10b981", markersize=11, label="Setor 300: Dosagem e Moagem"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#8b5cf6", markersize=11, label="Setor 400: Prensagem e Embalagem"),
]
ax.legend(handles=legend_elements, loc="upper left", frameon=True, facecolor="#ffffff", framealpha=0.9, fontsize=9.5)

ax.margins(0.12)
ax.axis("off")
plt.tight_layout()
plt.show()

---
### 7. Análise Crítica de Engenharia de Automação e Tomada de Decisão

A equivalência estrita dos comprimentos lineares ($W_{\text{Kruskal}} = W_{\text{Prim}} = 99{,}00\text{ metros}$) corrobora o teorema fundamental das Árvores Geradoras Mínimas em grafos conexos com pesos não-negativos. Contudo, sob o prisma da **Engenharia de Controle e Automação**, a seleção entre Kruskal e Prim decorre do estágio do ciclo de vida da planta industrial:

#### 1. Fase de Projeto Conceitual e Básico (*FEED - Front-End Engineering Design*)
- **Algoritmo Recomendado:** **Kruskal** $\mathcal{O}(|E| \log |E|)$.
- **Justificativa Industrial:** No início do projeto, a equipe de engenharia eletromecânica analisa a matriz global de distâncias do *layout* da planta em lote (*batch*). Como Kruskal é orientado a arestas, ele identifica de imediato os trechos mais curtos e econômicos de canaletas e leitos de tubulação, permitindo elaborar a lista de materiais (*Bill of Materials* - BOM) e o orçamento de CAPEX de ferragens e cabos mesmo antes da definição definitiva do ponto de ancoragem da sala de controle.

#### 2. Fase de Instalação e Montagem Eletromecânica em Campo (*Field Commissioning*)
- **Algoritmo Recomendado:** **Prim** $\mathcal{O}(|E| \log |V|)$.
- **Justificativa Industrial:** Na montagem de campo, a equipe de infraestrutura não instala canaletas em trechos dispersos e desconexos. O algoritmo de Prim expande a infraestrutura de forma estritamente conexa e contínua a partir do nó de entrada (`SILO-101`) ou da Sala de Controle Central. Isso viabiliza o lançamento contínuo de cabos de rede (PROFINET) e dutos de ar comprimido trecho a trecho, permitindo o comissionamento progressivo e seguro dos setores 100, 200, 300 e 400 sem deixar frentes de obra isoladas na fábrica.

#### 3. Diferenciação em Relação ao STP/RSTP de Redes Industriais (Aula 23)
É imperativo distinguir que a **MST de infraestrutura física** minimiza o somatório global dos comprimentos das canaletas civis e eletrocalhas. Já os protocolos de enlace da rede Ethernet industrial (como o **RSTP - IEEE 802.1w**, estudado na Aula 23) constroem dinamicamente uma **Árvore de Caminhos Mínimos (SPT)** ancorada no *Root Bridge*, cujo objetivo é minimizar o retardo de propagação de cada CLP até o servidor supervisório SCADA central, e não a metragem física de cobre ou fibra instalada.